# Model Retention Frontier v1

## Governance

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1` production composer.
- **Reference baseline**: `soxl_backtest_policy_reimplementation_v1`, the qlib-aligned SOXLBacktest buy/sell path.
- **Direct comparison baselines**: `model_retention_walkforward_gate_v1`, `cap10_wf2020_soxlbacktest_target_contract_v1`, `cap10_three_fold_bc_slice_screen_v1`, `cap10_three_fold_checkpoint_metric_replay_v2`, `state_objective_budget_usage_bc_official_replay_v1`.
- **Role**: `diagnose` and training handoff, not actor promotion.
- **Target window**: retention gate requires `wf_2020`, `wf_2022`, `wf_2023_2024`, plus YTD return alignment through the current safe replay window.
- **Changed layer**: `selection` / `model_selection` evidence routing.
- **External reference**: qlib-aligned SOXLBacktest buy/sell reference and existing cap10 target-contract screens.

## Hypothesis

The next useful training run should not start from another simple cap or renamed profile. The current frontier should point to the SOXLBacktest wf2020 target contract and three-fold BC/checkpoint path because existing retained actors fail either walk-forward return participation or YTD return alignment.

## Acceptance Criteria

1. Scan current model-selection summaries and retention gate thresholds.
2. Rank evidence by readiness for the next training run.
3. Keep SOXLBacktest as a reference baseline, not a promotable actor.
4. Write reproducible sidecars under this card.
5. End with a no-promotion decision unless a retained actor actually passes the gate.

## Closest Historical Cases

- **Closest success case**: `cap10_wf2020_soxlbacktest_target_contract_v1` created a non-leaking wf2020 target contract from the qlib-aligned SOXLBacktest replay.
- **Closest failure case**: `cap10_three_fold_checkpoint_metric_replay_v2` passed drawdown in all folds but failed wf2020 relative return by a large margin, so it cannot be retained.
- **Why this is not duplicate work**: this notebook does not rerun another target or checkpoint screen. It consolidates the current evidence into a ranked next-training frontier and prevents repeating failed low-participation/cap-only paths.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "soxl_quant").exists():
            return candidate
    raise RuntimeError("could not locate repo root containing src/soxl_quant")

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.model_retention_frontier import (
    scan_model_retention_frontier,
    write_retention_frontier_outputs,
)

OUTPUT_DIR = ROOT / "runtime" / "model_selection" / "card" / "model_retention_frontier_v1"
result = scan_model_retention_frontier(root=ROOT)
write_retention_frontier_outputs(result, output_dir=OUTPUT_DIR)
result["summary"]

{'generated_at': '2026-06-13T18:54:40.892836+00:00',
 'rows_scanned': 93,
 'completed_handoff_count': 63,
 'recommended_next_source': 'cap10_wf2020_bull_participation_diagnostic_v1',
 'recommended_next_action': 'Rerun checkpoint selection with backtest_profile bnh_return and relative_outperformance from the same qlib replay source before any new training.',
 'recommended_primary_blocker': 'missing_walkforward_evidence',
 'required_best_return_pct': 213.143462,
 'reference_return_pct': 132.077978,
 'max_walkforward_drawdown_pct': 20.0,
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024']}

In [2]:
frontier = pd.DataFrame(result["rows"])
frontier.head(12)[[
    "frontier_rank",
    "experiment",
    "primary_blocker",
    "failed_wf_windows",
    "worst_wf_max_drawdown_pct",
    "relative_return_gap_to_target_pp",
    "ytd_return_pct",
    "ytd_gap_to_best_pct",
    "next_action",
]]

,frontier_rank,experiment,primary_blocker,failed_wf_windows,worst_wf_max_drawdown_pct,relative_return_gap_to_target_pp,ytd_return_pct,ytd_gap_to_best_pct,next_action
0,1,cap10_wf2020_bull_participation_diagnostic_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,Rerun checkpoint selection with backtest_profi...
1,2,conservative_rebound_release_gate_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,revise_target_semantics_with_sparse_soxx_sprea...
2,3,constrained_checkpoint_selector_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,generate_reward_constrained_short_training_can...
3,4,fold_local_rebound_priority_conflict_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,revise_state_objective_target_semantics_before...
4,5,official_replay_bad_interval_feature_audit_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,build official-replay-gated risk-state checkpo...
5,6,official_replay_candidate_mechanism_pk_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,stress-test short-volshock action-cap overlay ...
6,7,quantbench_ensemble_current_walkforward_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,use_failure_columns_to_tune_bull_release_or_dr...
7,8,short_volshock_action_cap_stress_test_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,reward-timing objective should target at least...
8,9,short_volshock_drawdown_drift_feature_audit_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,screen drawdown-drift reward/checkpoint penalt...
9,10,sparse_release_drawdown_budget_guard_screen_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,move_drawdown_control_into_checkpoint_objectiv...


In [3]:
recommended = frontier.iloc[0].to_dict()
key_rows = frontier[frontier["experiment"].isin([
    "cap10_wf2020_soxlbacktest_target_contract_v1",
    "cap10_three_fold_checkpoint_metric_replay_v2",
    "state_objective_budget_usage_bc_official_replay_v1",
    "state_conditioned_overlay_gap_to_20_v1",
])][[
    "frontier_rank",
    "experiment",
    "primary_blocker",
    "failed_wf_windows",
    "wf_drawdown_gap_pct",
    "relative_return_gap_to_target_pp",
    "ytd_return_pct",
    "ytd_gap_to_soxlbacktest_pct",
    "ytd_gap_to_best_pct",
    "next_action",
]]
key_rows

,frontier_rank,experiment,primary_blocker,failed_wf_windows,wf_drawdown_gap_pct,relative_return_gap_to_target_pp,ytd_return_pct,ytd_gap_to_soxlbacktest_pct,ytd_gap_to_best_pct,next_action
40,41,cap10_wf2020_soxlbacktest_target_contract_v1,completed_handoff,,NaN,NaN,NaN,NaN,NaN,run_three_fold_bc_slice_screen_before_any_ppo
77,78,cap10_three_fold_checkpoint_metric_replay_v2,completed_handoff,wf_2020,0.00,322.75,NaN,NaN,NaN,inspect_failed_fold_drawdown_or_reward_drift_b...
78,79,state_objective_budget_usage_bc_official_repla...,completed_handoff,"wf_2020,wf_2023_2024",1.35,369.50,27.19,104.89,185.96,run_fold_local_ppo_only_if_ytd_and_wf_gap_is_n...
91,92,state_conditioned_overlay_gap_to_20_v1,completed_handoff,,NaN,NaN,NaN,NaN,NaN,


## Conclusion

- **Run result**: scanned the current model-selection card surface and wrote `retention_frontier.csv`, `summary.json`, and `conclusion.md`.
- **Decision**: no actor promotion and no actor retention from this notebook. The latest retention gate still has no promotable model.
- **Completed handoffs**: `49`; upstream target-contract, BC slice, checkpoint diagnostic, and sidecar-manifest source cards that already have downstream evidence are marked `completed_handoff` so they do not keep getting recommended as the next action.
- **Recommended next source**: `state_objective_budget_usage_bc_official_replay_v1`.
- **Recommended next action**: `run_fold_local_ppo_only_if_ytd_and_wf_gap_is_not_structural_else_revise_targets`. In the current evidence this is a joint drawdown/return gap, so the practical branch is target revision before more PPO, not another simple cap or renamed profile.
- **Key metrics**: `state_objective_budget_usage_bc_official_replay_v1` has `wf_2020` MaxDD `21.35%`, YTD return `27.19%`, SOXLBacktest gap `104.89pp`, and current-best gap `185.96pp`.
- **Risk note**: do not repeat the already-completed `cap10_wf2020_soxlbacktest_target_contract_v1 -> cap10_three_fold_bc_slice_screen_v1 -> cap10_three_fold_checkpoint_*` handoff as the next training step. It exposed the wf2020 participation gap but did not produce a retained actor.
- **Next action**: revise the fold-local target/reward so the BC/official replay path can raise YTD participation while keeping all WF windows under MaxDD `<20%`; only after that should a retained actor bundle be trained and sent back through the full retention gate.